In [1]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))
from omegaconf import OmegaConf
from functools import partial
import json
import glob

import numpy as np
import torch
from torchtune import config
from torchtune.data import padded_collate_packed
from torch.utils.data import DataLoader
import torch.nn.functional as F
import seaborn as sns
import pandas as pd


from dataset_classes import learning_levels_pfa_dataset, PackedOnTheFlyDataset
from evaluation.natural_language_levels_evaluation import (languages_levels_eval_dataset,
                                                            process_data,
                                                            compute_losses_per_level,
                                                            compute_losses_per_level_statistics,
nl_learning_levels_evaluation)
from training import SelfPredictionTrainingRecipeDistributed

In [2]:
import wandb

api = wandb.Api()
runs = api.runs("hidden-state-predictions/llama-3B-gumbel-layers")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: suurajperpeli (hidden-state-predictions) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
losses = ['next_token_losses', 'phi_losses0', 'latent_entropy0', 'latent_losses0']
levels = (0,1,2,3,4)

def eval_model(checkpoint_path, losses, levels):
    config_file = checkpoint_path + "/config.yaml"
    try:
        cfg = OmegaConf.load(config_file)
    except:
        return None
    if cfg.train_whole_model:
        print("load full model")
        cfg.checkpointer.checkpoint_dir = checkpoint_path
        cfg.checkpointer.checkpoint_files = ["torchtune_model_last.pt"]
    else:
        cfg.checkpointer.self_prediction_checkpoint_dir = checkpoint_path
        cfg.checkpointer.self_prediction_checkpoint = "hf_model_0001_None.pt"
        
    cfg.checkpointer.output_dir = "/home/woody/iwi5/iwi5368h/models/pfa_eval"
    cfg.train_from_scratch = False
    cfg.compile = False
    cfg.metric_logger._component_ = "torchtune.training.metric_logging.DiskLogger"
    recipe = SelfPredictionTrainingRecipeDistributed(cfg=cfg)
    recipe.setup(cfg=cfg)

    dataset = languages_levels_eval_dataset(recipe._tokenizer,location='/home/hpc/iwi5/iwi5368h/predicting-hidden-states/data/natural_language_levels',)


    
    recipe._model.eval()
    datapoints = process_data(recipe,num_datapoints=50,dataset=dataset,batch_size=4,)
    lvll = compute_losses_per_level(datapoints, 
                                    losses=losses, 
                                    levels=levels,
                                    filter_out_spaces=False,
                                    level_key='level',)
    lvlls = compute_losses_per_level_statistics(lvll,
                                                losses=losses,
                                                levels=levels,)
    for k, v in lvlls.items():
        for l, s in v.items():
            for f, value in s.items():
                if type(value) != int:
                    lvlls[k][l][f] = float(value)
    return datapoints, lvlls

In [4]:
layers_vs_stats = []
for run in runs[:4]:
    checkpoint_base = "/home/woody/iwi5/iwi5368h/models/llama_3B_PHi/learning_levels_sweep_"  # llama-0.1B
    checkpoint_path = checkpoint_base + run.id
    
    _exists = glob.glob(checkpoint_path)
    print(f'{run.id} exists?: {_exists!=[]}')
    if _exists == []:
        continue
    
    datapoints, stats = eval_model(checkpoint_path, losses, levels)
    layers_vs_stats.append({
        'layer': int(run.name.split('-')[3]),
        'seed': int(run.name.split('-')[1]),
        'statistics': stats, 
        'run_id': str(run.id)} )
with open(f"/home/woody/iwi5/iwi5368h/evaluations/pfa_evals_3B/layers-run-gumbel.json", 'w') as f:
    print('saving run')
    json.dump(layers_vs_stats, f)

3j1t3kbo exists?: False
imc5f6j0 exists?: False
6w0xj5vh exists?: True


RuntimeError: bf16 precision was requested but not available on this hardware. Please use fp32 precision instead.